# HiveMind benchmark on a Colab A100

Runtime -> Change runtime type -> **A100 GPU** (High-RAM if offered). Run the cells top to bottom.
Same model + same llama-server for every row: `hivemind_duo`, `raw`, `aider`.

The current `dev` branch is pushed (commit `dd88878`), so Colab clones the real
HiveMind tree instead of uploading a bundle. Smoke first, then full. For the
5-task DeepSWE subset use `bench/external/deepswe_hivemind.py --verifier modal`
after `setup`/`serve`; that grades in Modal, not local Docker.


In [ ]:
# 1) GPU + disk check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /root | tail -1


In [ ]:
# 2) Clone the pushed dev branch and the benchmark bootstrap
![ -d /root/HiveMind_llama/.git ] || git clone --branch dev https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cp /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!ls -l /root/rent_bootstrap.sh


In [ ]:
# 3) Setup (~15-30 min: Python 3.14, llama.cpp CUDA, 20 GB model, aider)
!bash /root/rent_bootstrap.sh setup


In [ ]:
# 4) Start the HiveMind engine in the background
!bash /root/rent_bootstrap.sh serve


In [ ]:
# 5) Smoke test: 3 tasks x 3 conditions. READ the output (pass/fail, tokens > 0, wall time)
!bash /root/rent_bootstrap.sh smoke


In [ ]:
# 6) Full run (detached in tmux). Optionally: HM_TASKS=bench/tasks_polyglot after `polyglot`
import os
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = 'hivemind_duo,raw,aider'
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 7) Progress (re-run as often as you like)
!bash /root/rent_bootstrap.sh status


In [ ]:
# 8) Save results to Drive (do this when status shows FINISHED)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hivemind_bench && cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/
!cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ || true
